In [4]:
!pip install transformers

In [5]:
import pandas as pd

df = pd.read_csv("/content/Tweets.csv")

df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [6]:
# Keep only positive and negative tweets
df_binary = df[df["airline_sentiment"].isin(["positive", "negative"])].copy()

# Convert dataset labels to the same labels used by the Transformer
df_binary["true_label"] = df_binary["airline_sentiment"].map({
    "positive": "POSITIVE",
    "negative": "NEGATIVE"
})

print(df_binary["true_label"].value_counts())

true_label
NEGATIVE    9178
POSITIVE    2363
Name: count, dtype: int64


In [7]:
sample = df_binary.sample(n=150, random_state=42)

texts = sample["text"].tolist()
true_labels = sample["true_label"].tolist()

In [8]:
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [9]:
predictions = sentiment_pipeline(texts)

predicted_labels = [result["label"] for result in predictions]

print(predicted_labels[:10])

['NEGATIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE']


In [10]:
correct = 0

for true, predicted in zip(true_labels, predicted_labels):
    if true == predicted:
        correct += 1

accuracy = correct / len(true_labels)

print("Correct predictions:", correct)
print("Total predictions:", len(true_labels))
print("Fraction correct:", accuracy)

Correct predictions: 132
Total predictions: 150
Fraction correct: 0.88


In [11]:
for i in range(5):
    print("Text:", texts[i])
    print("True label:", true_labels[i])
    print("Predicted label:", predicted_labels[i])
    print("-" * 60)

Text: @USAirways They charged me for a flight they Cancelled Flightled, unbelievable and unheard of
True label: NEGATIVE
Predicted label: NEGATIVE
------------------------------------------------------------
Text: @JetBlue great flight! Great view! :-) http://t.co/Yxn00pnOav
True label: POSITIVE
Predicted label: POSITIVE
------------------------------------------------------------
Text: @united they're not, actually. gate agent was so rude. now standing in a line waiting for reFlight Booking Problems. missed the only flight to STI. awful.
True label: NEGATIVE
Predicted label: NEGATIVE
------------------------------------------------------------
Text: @AmericanAir No worries they called back 4 hrs Late Flightr while I was asleep and took an additional $200 fee. So by AA standards everything's gr8
True label: NEGATIVE
Predicted label: NEGATIVE
------------------------------------------------------------
Text: @united thank you. There was one here a few months ago, but none now. Weird you

Attention allows the Transformer to consider how words relate to other words in the sentence, including words that may be far apart. Unlike Bag-of-Words, which mainly counts individual word occurrences and ignores word order and context, attention helps the model understand the context and relationships between words